
VoiceCare PHC — IndicTrans2 API Server for Google Colab



In [2]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run this cell first, then RESTART RUNTIME, then run Cell 2 onwards

%%capture
!pip install flask flask-cors pyngrok openai-whisper gtts
!pip install transformers==4.53.2 sentencepiece sacremoses
!pip install bitsandbytes scipy accelerate nltk
!pip install python-multipart

!git clone https://github.com/AI4Bharat/IndicTrans2.git
%cd /content/IndicTrans2/huggingface_interface

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!pip install --editable ./
%cd ..

import nltk
nltk.download("punkt", quiet=True)

print("✅ Cell 1 done — now RESTART RUNTIME (Runtime → Restart session), then run Cell 2")

In [1]:
# ── Cell 2: Load models ───────────────────────────────────────────────────────
# Run AFTER restarting runtime

import sys
import os
from google.colab import userdata

# Authenticate with HuggingFace
hf_token = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=hf_token)

sys.path.insert(0, "/content/IndicTrans2/huggingface_interface/IndicTransToolkit")
os.chdir("/content/IndicTrans2/huggingface_interface")

import torch
import whisper
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

try:
    from IndicTransToolkit import IndicProcessor
    print("✅ Imported IndicProcessor from IndicTransToolkit (top-level)")
except ImportError:
    try:
        from IndicTransToolkit.processor import IndicProcessor
        print("✅ Imported IndicProcessor from IndicTransToolkit.processor")
    except ImportError:
        from indicProcessor import IndicProcessor
        print("✅ Imported IndicProcessor from indicProcessor")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Whisper STT
print("Loading Whisper base...")
whisper_model = whisper.load_model("base", device=DEVICE)
print("✅ Whisper loaded")

# IndicTrans2 — Indic → English
print("Loading IndicTrans2 Indic→EN (200M)...")
indic_en_ckpt = "ai4bharat/indictrans2-indic-en-dist-200M"
indic_en_tokenizer = AutoTokenizer.from_pretrained(
    indic_en_ckpt, trust_remote_code=True, token=hf_token
)
indic_en_model = AutoModelForSeq2SeqLM.from_pretrained(
    indic_en_ckpt, trust_remote_code=True, low_cpu_mem_usage=True, token=hf_token
).to(DEVICE).half().eval()
print("✅ Indic→EN loaded")

# IndicTrans2 — English → Indic
print("Loading IndicTrans2 EN→Indic (200M)...")
en_indic_ckpt = "ai4bharat/indictrans2-en-indic-dist-200M"
en_indic_tokenizer = AutoTokenizer.from_pretrained(
    en_indic_ckpt, trust_remote_code=True, token=hf_token
)
en_indic_model = AutoModelForSeq2SeqLM.from_pretrained(
    en_indic_ckpt, trust_remote_code=True, low_cpu_mem_usage=True, token=hf_token
).to(DEVICE).half().eval()
print("✅ EN→Indic loaded")

ip = IndicProcessor(inference=True)
print("\n✅ All models loaded! Proceed to Cell 3.")

✅ Imported IndicProcessor from IndicTransToolkit (top-level)
Device: cuda
Loading Whisper base...
✅ Whisper loaded
Loading IndicTrans2 Indic→EN (200M)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

... (22 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

✅ EN→Indic loaded

✅ All models loaded! Proceed to Cell 3.


In [2]:
# ── Cell 3: Helper functions ──────────────────────────────────────────────────

LANG_CODE_MAP = {
    "Telugu":    "tel_Telu",
    "Hindi":     "hin_Deva",
    "Tamil":     "tam_Taml",
    "Kannada":   "kan_Knda",
    "Bengali":   "ben_Beng",
    "Gujarati":  "guj_Gujr",
    "Marathi":   "mar_Deva",
    "Punjabi":   "pan_Guru",
    "Odia":      "ory_Orya",
    "Malayalam": "mal_Mlym",
}

GTTS_LANG_MAP = {
    "Telugu":    "te",
    "Hindi":     "hi",
    "Tamil":     "ta",
    "Kannada":   "kn",
    "Bengali":   "bn",
    "Gujarati":  "gu",
    "Marathi":   "mr",
    "Punjabi":   "pa",
    "Odia":      "or",
    "Malayalam": "ml",
    "English":   "en",
}


def batch_translate(sentences, src_lang, tgt_lang, model, tokenizer, batch_size=4):
    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i: i + batch_size]
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(
            batch, truncation=True, padding="longest",
            return_tensors="pt", return_attention_mask=True
        ).to(DEVICE)
        with torch.no_grad():
            tokens = model.generate(
                **inputs, use_cache=True, min_length=0,
                max_length=256, num_beams=4, num_return_sequences=1
            )
        decoded = tokenizer.batch_decode(
            tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )
        translations += ip.postprocess_batch(decoded, lang=tgt_lang)
        del inputs
        torch.cuda.empty_cache()
    return translations


def transcribe_audio(audio_path, language_name):
    lang_hint = GTTS_LANG_MAP.get(language_name, None)
    result = whisper_model.transcribe(audio_path, language=lang_hint, task="transcribe")
    return result["text"].strip()


def translate_to_english(text, src_language_name):
    src_code = LANG_CODE_MAP.get(src_language_name, "tel_Telu")
    return batch_translate([text], src_code, "eng_Latn",
                           indic_en_model, indic_en_tokenizer)[0]


def translate_to_local(english_text, tgt_language_name):
    tgt_code = LANG_CODE_MAP.get(tgt_language_name, "tel_Telu")
    return batch_translate([english_text], "eng_Latn", tgt_code,
                           en_indic_model, en_indic_tokenizer)[0]


def text_to_speech(text, language_name, output_path):
    from gtts import gTTS
    lang_code = GTTS_LANG_MAP.get(language_name, "en")
    gTTS(text=text, lang=lang_code, slow=False).save(output_path)
    return output_path

print("✅ Helper functions ready. Proceed to Cell 4.")


✅ Helper functions ready. Proceed to Cell 4.


In [3]:
# ── Cell 4: Flask API ─────────────────────────────────────────────────────────

import uuid
import base64
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

UPLOAD_DIR = "/tmp/vc_uploads"
OUTPUT_DIR = "/tmp/vc_outputs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "device": DEVICE})


@app.route("/voice-process", methods=["POST"])
def voice_process():
    try:
        if "audio_file" not in request.files:
            return jsonify({"error": "No audio_file"}), 400

        language   = request.form.get("language", "Telugu")
        uid        = str(uuid.uuid4())[:8]
        audio_path = f"{UPLOAD_DIR}/{uid}_in.wav"
        request.files["audio_file"].save(audio_path)

        local_text = transcribe_audio(audio_path, language)

        if not local_text.strip():
            return jsonify({
                "error": "Could not transcribe audio. Please speak clearly and try again.",
                "local_text": "",
                "english_text": "",
                "local_back_text": ""
            }), 422

        english_text = translate_to_english(local_text, language)
        local_back   = translate_to_local(english_text, language)

        en_path    = f"{OUTPUT_DIR}/{uid}_en.mp3"
        local_path = f"{OUTPUT_DIR}/{uid}_local.mp3"
        text_to_speech(english_text, "English", en_path)
        text_to_speech(local_back,   language,  local_path)

        en_b64    = base64.b64encode(open(en_path,    "rb").read()).decode()
        local_b64 = base64.b64encode(open(local_path, "rb").read()).decode()

        for p in [audio_path, en_path, local_path]:
            try: os.remove(p)
            except: pass

        return jsonify({
            "success":           True,
            "language":          language,
            "local_text":        local_text,
            "english_text":      english_text,
            "local_back_text":   local_back,
            "english_audio_b64": en_b64,
            "local_audio_b64":   local_b64,
        })
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/translate-text", methods=["POST"])
def translate_text():
    data      = request.json
    text      = data.get("text", "")
    language  = data.get("src_language", "Telugu")
    direction = data.get("direction", "to_english")
    result    = translate_to_english(text, language) if direction == "to_english" \
                else translate_to_local(text, language)
    return jsonify({"success": True, "result": result})

print("✅ Flask app ready. Proceed to Cell 5.")

✅ Flask app ready. Proceed to Cell 5.


In [4]:
# ── Cell 5: Start ngrok + Flask ───────────────────────────────────────────────

from pyngrok import ngrok

# Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3BNYGQMrbRNWcQDEGHibUqTOiCS_7woCGumGnwnT8QDPi5LnY"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(5000, bind_tls=True).public_url

print("\n" + "="*60)
print(f"  VoiceCare API LIVE at: {public_url}")
print(f"\n  Add to backend/.env:")
print(f"  VOICE_API_URL={public_url}")
print("="*60 + "\n")

app.run(port=5000, use_reloader=False)


  VoiceCare API LIVE at: https://nonactinically-carneous-lesley.ngrok-free.dev

  Add to backend/.env:
  VOICE_API_URL=https://nonactinically-carneous-lesley.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 05:57:01] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 05:57:02] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 06:00:11] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 06:00:13] "POST /translate-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 06:00:14] "POST /translate-text HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 06:00:29] "POST /voice-process HTTP/1.1" 500 -
